In [0]:
from pyspark.sql.functions import *

### Customer Analytics Model

In [0]:
def build_customer_analytics():

    customers_df = spark.table(
        "cicd_dev.silver.customers"
    )

    sales_df = spark.table(
        "cicd_dev.silver.sales"
    )

    return (
        customers_df.alias("c")
        .join(
            sales_df.alias("s"),
            "customer_id",
            "left"
        )
        .groupBy(
            "customer_id",
            "customer_name",
            "city"
        )
        .agg(
            count("sale_id").alias("total_orders"),
            sum("sales_amount").alias("total_sales")
        )
    )

### Product Analytics Model

In [0]:
def build_product_analytics():

    products_df = spark.table(
        "cicd_dev.silver.products"
    )

    sales_df = spark.table(
        "cicd_dev.silver.sales"
    )

    return (
        products_df.alias("p")
        .join(
            sales_df.alias("s"),
            "product_id",
            "left"
        )
        .groupBy(
            "product_id",
            "product_name",
            "category"
        )
        .agg(
            sum("quantity").alias(
                "total_quantity_sold"
            ),
            sum("sales_amount").alias(
                "total_revenue"
            )
        )
    )

### Sales Analytics Model

In [0]:
def build_sales_analytics():

    sales_df = spark.table(
        "cicd_dev.silver.sales"
    )

    return (
        sales_df
        .groupBy("sale_date")
        .agg(
            count("sale_id").alias(
                "total_orders"
            ),
            sum("sales_amount").alias(
                "total_revenue"
            )
        )
    )

### Inventory Analytics Model

In [0]:
def build_inventory_analytics():

    inventory_df = spark.table(
        "cicd_dev.silver.inventory"
    )

    products_df = spark.table(
        "cicd_dev.silver.products"
    )

    return (
        inventory_df.alias("i")
        .join(
            products_df.alias("p"),
            "product_id"
        )
        .withColumn(
            "stock_status",
            when(
                col("stock_quantity") < 100,
                "LOW"
            ).otherwise("AVAILABLE")
        )
    )

In [0]:
gold_models = {

    "customer_analytics":
        build_customer_analytics,

    "product_analytics":
        build_product_analytics,

    "sales_analytics":
        build_sales_analytics,

    "inventory_analytics":
        build_inventory_analytics

}

In [0]:
metadata_df = spark.table(
    "cicd_dev.gold.gold_metadata"
)

display(metadata_df)

In [0]:
from pyspark.sql.functions import current_timestamp

metadata = metadata_df.collect()

for row in metadata:

    model_name = row["model_name"]
    gold_table = row["gold_table"]

    try:

        print(
            f"Building {model_name}"
        )

        df = gold_models[
            model_name
        ]()

        target_table = (
            f"cicd_dev.gold.{gold_table}"
        )

        df.write \
          .format("delta") \
          .mode("overwrite") \
          .saveAsTable(target_table)

        record_count = df.count()

        audit_df = spark.createDataFrame(
            [
                (
                    gold_table,
                    record_count,
                    "SUCCESS"
                )
            ],
            [
                "table_name",
                "record_count",
                "status"
            ]
        ).select(
            "*",
            current_timestamp().alias(
                "load_time"
            )
        )

        audit_df.write.mode(
            "append"
        ).saveAsTable(
            "cicd_dev.gold.gold_audit"
        )

        print(
            f"Loaded {gold_table}"
        )

    except Exception as e:

        print(
            f"Failed {gold_table}"
        )

        audit_df = spark.createDataFrame(
            [
                (
                    gold_table,
                    0,
                    f"FAILED : {str(e)}"
                )
            ],
            [
                "table_name",
                "record_count",
                "status"
            ]
        ).select(
            "*",
            current_timestamp().alias(
                "load_time"
            )
        )

        audit_df.write.mode(
            "append"
        ).saveAsTable(
            "cicd_dev.gold.gold_audit"
        )

In [0]:
%sql
SHOW TABLES IN cicd_dev.gold;

In [0]:
%sql
SELECT *
FROM cicd_dev.gold.gold_audit
ORDER BY load_time DESC;